# Week 3 Lab: Distributed Training with DeepSpeed
## Overview
In this lab, you will explore the practical implementation of distributed training. We move from standard PyTorch Data Parallelism to DeepSpeed's ZeRO (Zero Redundancy Optimizer) stages, pipeline parallelism, and advanced memory optimizations.

## Part 1: Environment Setup
Goal: Install dependencies and verify GPU communication.

In [2]:
# 1. Install Dependencies
!pip install torch
!pip install deepspeed transformers datasets accelerate evaluate

In [3]:
import torch
import deepspeed
import transformers
import os

# 2. Verify GPU Environment
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

CUDA Available: True
GPU Count: 2
GPU 0: NVIDIA RTX 5880 Ada Generation
GPU 1: NVIDIA RTX 5880 Ada Generation


In [ ]:
# 3. Create a workspace directory
os.makedirs("ds_lab", exist_ok=True)

## Part 2: Data Parallelism Basics
Goal: Establish a baseline and understand Native PyTorch DDP.
### 2.1 Single GPU Baseline
We will use a helper script to train a GPT-2 model on a single GPU to measure baseline memory and throughput.

In [14]:
%%writefile ds_lab/baseline.py
import torch
from transformers import GPT2LMHeadModel, GPT2Config, AutoTokenizer
from datasets import load_dataset
import time

# Setup
device = "cuda:0"
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load tiny subset of WikiText
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_loader = torch.utils.data.DataLoader(tokenized_datasets['input_ids'], batch_size=4)

model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Profile
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

model.train()
for i, batch in enumerate(train_loader):
    if i >= 10: break # Run 10 steps for profiling
    inputs = torch.stack(batch).to(device)
    outputs = model(inputs, labels=inputs)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

end_time = time.time()
max_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Baseline Throughput: {10 * 4 / (end_time - start_time):.2f} samples/sec")
print(f"Peak Memory: {max_mem:.2f} GB")

Overwriting ds_lab/baseline.py


In [4]:
# !source ~/anaconda3/etc/profile.d/conda.sh
# !conda activate nlplab
!python ds_lab/baseline.py

CUDA_VISIBLE_DEVICES: None
torch.cuda.device_count(): 2
Loading weights: 100%|█████████████████████| 292/292 [00:00<00:00, 12000.16it/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Baseline Throughput: 16.05 samples/sec
Peak Memory: 12.11 GB


### 2.2 Native PyTorch DDP (Distributed Data Parallel)
In DDP, every GPU keeps a full copy of the model. Gradients are averaged across GPUs during the backward pass.

In [5]:
%%writefile ds_lab/torch_ddp.py
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from transformers import GPT2LMHeadModel, AutoTokenizer
from datasets import load_dataset
import os
import time

def setup():
    dist.init_process_group("nccl")
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))

def cleanup():
    dist.destroy_process_group()

def train():
    setup()
    local_rank = int(os.environ["LOCAL_RANK"])
    
    model_name = "gpt2-medium"
    model = GPT2LMHeadModel.from_pretrained(model_name).to(local_rank)
    ddp_model = DDP(model, device_ids=[local_rank])
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")
    
    # DDP requires a DistributedSampler to ensure each GPU sees different data
    tokenized = dataset.map(lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=512), batched=True)
    sampler = DistributedSampler(tokenized, shuffle=False)
    loader = torch.utils.data.DataLoader(tokenized['input_ids'], batch_size=4, sampler=sampler)

    optimizer = torch.optim.AdamW(ddp_model.parameters(), lr=5e-5)

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    ddp_model.train()
    for i, batch in enumerate(loader):
        if i >= 10: break
        inputs = torch.stack(batch).to(local_rank)
        outputs = ddp_model(inputs, labels=inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    if local_rank == 0:
        max_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"DDP Throughput: {10 * 4 * dist.get_world_size() / (time.time() - start_time):.2f} samples/sec")
        print(f"DDP Peak Memory: {max_mem:.2f} GB")

    cleanup()

if __name__ == "__main__":
    train()

Overwriting ds_lab/torch_ddp.py


In [7]:
!# Source - https://stackoverflow.com/a/20357035
# Posted by Alec Teal, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-07, License - CC BY-SA 4.0

!export LD_LIBRARY_PATH="/usr/local/lib64/:$LD_LIBRARY_PATH"

In [11]:
!torchrun --nproc_per_node=2 ds_lab/torch_ddp.py

W0407 10:42:29.096000 2416801 site-packages/torch/distributed/run.py:851] 
W0407 10:42:29.096000 2416801 site-packages/torch/distributed/run.py:851] *****************************************
W0407 10:42:29.096000 2416801 site-packages/torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0407 10:42:29.096000 2416801 site-packages/torch/distributed/run.py:851] *****************************************
[rank 1] Loading model gpt2 (local_rank=1, cuda_device_count=2, NCCL_P2P_DISABLE=1)
[rank 0] Loading model gpt2 (local_rank=0, cuda_device_count=2, NCCL_P2P_DISABLE=1)
Loading weights: 100%|█████████████████████| 148/148 [00:00<00:00, 12052.83it/s]
[rank 1] Model weights loaded on CPU
[rank 1] Model moved to CUDA
[rank 0] Model weights loaded on CPU
[rank 0] Model moved to CUDA
[rank 0] Model loaded and wr

## Part 3: DeepSpeed ZeRO Stages
Goal: Implement ZeRO-1, 2, and 3 using DeepSpeed.

To understand why memory changes, you must understand the "Memory Tax" of the Adam optimizer in 16-bit training. For a model with Ψ parameters:
- Static Memory: Weights (2 bytes/param) + Gradients (2 bytes/param) = 4Ψ
- Optimizer States (Adam): FP32 Copy of Weights (4b) + Momentum (4b) + Variance (4b) = 12Ψ
- Total: 16Ψ bytes. (e.g., a 1B parameter model needs 16GB just for states/weights, before activations).

### 3.1 Creating the DeepSpeed Training Script
We will create a generic script that takes a DeepSpeed configuration file.

In [12]:
%%writefile ds_lab/ds_train.py
import torch
import deepspeed
from transformers import GPT2LMHeadModel, AutoTokenizer
from datasets import load_dataset
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--deepspeed_config', type=str)
parser.add_argument('--local_rank', type=int)
args = parser.parse_args()

model_name = "gpt2-medium"
model = GPT2LMHeadModel.from_pretrained(model_name)

# Dataset setup
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:5%]")
tokenized = dataset.map(lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=512), batched=True)
train_loader = torch.utils.data.DataLoader(tokenized['input_ids'], batch_size=4)

# Initialize DeepSpeed
print(f"\n\nLoad Model\n\n")
model_engine, optimizer, _, _ = deepspeed.initialize(
    args=args,
    model=model,
    model_parameters=model.parameters()
)

# Training loop
print(f"\n\nTraining...\n\n")
for i, batch in enumerate(train_loader):
    if i >= 10: break
    if model_engine.local_rank == 0:
        print(f"Step {i} Max Memory Allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    inputs = torch.stack(batch).to(model_engine.local_rank)
    outputs = model_engine(inputs, labels=inputs)
    loss = outputs.loss
    model_engine.backward(loss)
    model_engine.step()
if model_engine.local_rank == 0:
    print(f"Max Memory Allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Overwriting ds_lab/ds_train.py


### 3.2 ZeRO-1 (Optimizer State Partitioning)

Every GPU keeps a full copy of the model weights and gradients. However, the Optimizer States (the 12Ψ part) are divided equally among GPUs. After the backward pass, gradients are reduced across GPUs. Each GPU updates only its slice of the optimizer states and its slice of the FP32 weights. It then broadcasts the updated slice back to all other GPUs.

**Memory Reduction**: From 16Ψ down to 4Ψ+ 12Ψ/N (where N is number of GPUs).

In [13]:
%%writefile ds_lab/ds_config_z1.json
{
    "train_micro_batch_size_per_gpu": 4,
    "optimizer": {
        "type": "AdamW",
        "params": { "lr": 5e-5 }
    },
    "zero_optimization": {
        "stage": 1
    },
    "fp16": { "enabled": true }
}

Overwriting ds_lab/ds_config_z1.json


### 3.3 ZeRO-2 (Gradient Partitioning)

In addition to sharding optimizer states, we now shard the Gradients. As gradients are calculated during the backward pass, they are "Reduced-Scattered" to the GPUs responsible for those specific parameters. Once a GPU has the averaged gradient for its shard, it deletes the gradients for all other shards.

**Memory Reduction:** From 16Ψ down to 2Ψ+ 14Ψ/N.

In [14]:
%%writefile ds_lab/ds_config_z2.json
{
    "train_micro_batch_size_per_gpu": 4,
    "optimizer": {
        "type": "AdamW",
        "params": { "lr": 5e-5 }
    },
    "zero_optimization": {
        "stage": 2
    },
    "fp16": { "enabled": true }
}

Overwriting ds_lab/ds_config_z2.json


### 3.4 ZeRO-3 (Parameter Partitioning)

ZeRO-3 allows training models larger than a single GPU's memory by sharding weights. The Weights themselves are now sharded across GPUs. No single GPU holds the full model.

- Forward Pass: GPU 0 asks all other GPUs for their weight shards to "materialize" a specific layer. Once the layer is computed, the shards are deleted.
- Backward Pass: Same "on-demand" gathering for weight gradients.
- **Memory Reduction:** Linear reduction: 16Ψ/N.
- **The Trade-off:** This significantly increases communication (inter-GPU traffic) because weights are being moved constantly.

In [15]:
%%writefile ds_lab/ds_config_z3.json
{
    "train_micro_batch_size_per_gpu": 4,
    "optimizer": {
        "type": "AdamW",
        "params": { "lr": 5e-5 }
    },
    "zero_optimization": {
        "stage": 3,
        "offload_param": { "device": "cpu" }
    },
    "fp16": { "enabled": true }
}

Overwriting ds_lab/ds_config_z3.json


In [17]:
!deepspeed --num_gpus=2 ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z1.json

[2026-04-07 10:48:20,032] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-04-07 10:48:20,032] [INFO] [runner.py:630:main] cmd = /home/khang.nhat/anaconda3/envs/nlplab/bin/python3.11 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z1.json
[2026-04-07 10:48:23,041] [INFO] [launch.py:162:main] WORLD INFO DICT: {'localhost': [0, 1]}
[2026-04-07 10:48:23,041] [INFO] [launch.py:168:main] nnodes=1, num_local_procs=2, node_rank=0
[2026-04-07 10:48:23,041] [INFO] [launch.py:179:main] global_rank_mapping=defaultdict(<class 'list'>, {'localhost': [0, 1]})
[2026-04-07 10:48:23,041] [INFO] [launch.py:180:main] dist_world_size=2
[2026-04-07 10:48:23,041] [INFO] [launch.py:184:main] Setting CUDA_VISIBLE_DEVICES=0,1
[2026-04-07 10:48:23,041] [INFO] [launch.p

In [18]:
!deepspeed --num_gpus=2 ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z2.json

[2026-04-07 10:51:20,829] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-04-07 10:51:20,829] [INFO] [runner.py:630:main] cmd = /home/khang.nhat/anaconda3/envs/nlplab/bin/python3.11 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z2.json
[2026-04-07 10:51:23,813] [INFO] [launch.py:162:main] WORLD INFO DICT: {'localhost': [0, 1]}
[2026-04-07 10:51:23,813] [INFO] [launch.py:168:main] nnodes=1, num_local_procs=2, node_rank=0
[2026-04-07 10:51:23,813] [INFO] [launch.py:179:main] global_rank_mapping=defaultdict(<class 'list'>, {'localhost': [0, 1]})
[2026-04-07 10:51:23,813] [INFO] [launch.py:180:main] dist_world_size=2
[2026-04-07 10:51:23,813] [INFO] [launch.py:184:main] Setting CUDA_VISIBLE_DEVICES=0,1
[2026-04-07 10:51:23,814] [INFO] [launch.p

In [19]:
!deepspeed --num_gpus=2 ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z3.json

[2026-04-07 10:51:59,872] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-04-07 10:51:59,873] [INFO] [runner.py:630:main] cmd = /home/khang.nhat/anaconda3/envs/nlplab/bin/python3.11 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info ds_lab/ds_train.py --deepspeed_config ds_lab/ds_config_z3.json
[2026-04-07 10:52:02,934] [INFO] [launch.py:162:main] WORLD INFO DICT: {'localhost': [0, 1]}
[2026-04-07 10:52:02,934] [INFO] [launch.py:168:main] nnodes=1, num_local_procs=2, node_rank=0
[2026-04-07 10:52:02,934] [INFO] [launch.py:179:main] global_rank_mapping=defaultdict(<class 'list'>, {'localhost': [0, 1]})
[2026-04-07 10:52:02,934] [INFO] [launch.py:180:main] dist_world_size=2
[2026-04-07 10:52:02,934] [INFO] [launch.py:184:main] Setting CUDA_VISIBLE_DEVICES=0,1
[2026-04-07 10:52:02,934] [INFO] [launch.p